# EmpathyTransformer — Training
Train dari 37.5K quotes + conversation pairs.
Model: ~78M params, Transformer dari scratch.

Runtime → Change runtime type → **T4 GPU**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

In [ ]:
import os, shutil

# Ganti path ini sesuai folder Drive kamu
DRIVE_PATH = '/content/drive/MyDrive/aitest'

if not os.path.exists(DRIVE_PATH):
    # Upload dulu folder aitest/ ke Drive kamu
    # Baris ini bikin foldernya
    os.makedirs(DRIVE_PATH, exist_ok=True)
    print(f'Buat folder {DRIVE_PATH} dulu. Upload semua file dari aitest/ ke sini.')
else:
    # Copy ke Colab environment (lebih cepet I/O)
    if os.path.exists('/content/aitest'):
        shutil.rmtree('/content/aitest')
    shutil.copytree(DRIVE_PATH, '/content/aitest')
    print(f'Copied {DRIVE_PATH} to /content/aitest')
    print('Files:', os.listdir('/content/aitest/data/'))

In [ ]:
%%capture
!pip install torch tokenizers tqdm
print('Dependencies installed')

## 1. Train Tokenizer

In [ ]:
import os
os.chdir('/content/aitest')

DATA_PATH = 'data/train_mixed.jsonl'
TOK_PATH = 'tokenizer.json'

# Train BPE tokenizer (vocab 16K — ringan & cepet)
!python tokenizer.py --data {DATA_PATH} --vocab-size 16384 --save {TOK_PATH}

# Verify
from tokenizer import load_tokenizer
tok = load_tokenizer(TOK_PATH)
print(f'Vocab size: {tok.get_vocab_size()}')
print(f'Sample encode: "Hello world" → {tok.encode("Hello world").ids[:10]}')

## 2. Train Model

**Epochs:** 3-5 cukup untuk dataset 37.5K.
**Batch size:** 16 di T4 16GB.
**Waktu:** ~2-4 jam tergantung epochs.

In [ ]:
!python train.py \
    --data {DATA_PATH} \
    --tokenizer {TOK_PATH} \
    --epochs 10 \
    --batch-size 8 \
    --grad-accum 8 \
    --lr 1e-3 \
    --amp \
    --grad-checkpoint \
    --empathy \
    --checkpoint-dir ./checkpoints \
    --log-every 5 \
    --save-every 500 \
    --val-every 250

print('\nTraining selesai!')

## 3. Test Model

In [ ]:
import sys
sys.path.insert(0, '/content/aitest')

from inference import load_model, generate_text
from tokenizer import load_tokenizer
import torch

# Load best model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = load_model('checkpoints/best.pt', device)
tokenizer = load_tokenizer('tokenizer.json')
print(f'Model loaded on {device}')

# Test prompts
prompts = [
    'How are you today?',
    'I feel really lonely...',
    'What is the meaning of life?',
    'I am struggling with this breakup',
    'Tell me something wise',
]

for p in prompts:
    resp = generate_text(model, tokenizer, p, max_new_tokens=50,
                         temperature=0.7, top_k=30, top_p=0.85, device=device)
    print(f'\nYOU: {p}')
    print(f'AI:  {resp}')

## 4. Save Model ke Drive

In [ ]:
import shutil

# Copy model + tokenizer back to Drive
checkpoint_files = ['checkpoints/best.pt', 'checkpoints/final.pt', 'tokenizer.json']
for f in checkpoint_files:
    src = f'/content/aitest/{f}'
    dst = f'{DRIVE_PATH}/{f}'
    if os.path.exists(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        print(f'Saved {src} → {dst}')

print('\nSemua file tersimpan di Drive!')

## Interactive Mode

Chat langsung dengan model setelah training.

In [ ]:
# Interactive chat — jalankan setelah training
from inference import generate_text

print('\n=== EmpathyTransformer Chat ===')
print('Type "quit" to exit.\n')

while True:
    prompt = input('YOU: ').strip()
    if prompt.lower() in ('quit', 'exit', 'q'):
        break
    if not prompt:
        continue

    resp = generate_text(model, tokenizer, prompt,
                         max_new_tokens=100,
                         temperature=0.7,
                         top_k=30,
                         top_p=0.85,
                         device=device)
    print(f'AI:  {resp}\n')